# **Explore Advanced Retrievers in LlamaIndex (OpenRouter)**

Estimated time needed: **60** minutes

This comprehensive lab demonstrates advanced retrieval techniques in LlamaIndex using **OpenRouter** as the LLM provider. You'll learn core retrievers, advanced retrievers, and sophisticated fusion techniques that power modern RAG applications.

Through hands-on examples, you'll master the art of building intelligent information retrieval systems that can handle complex queries, combine multiple search strategies, and deliver precise results for production RAG applications.

## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
            <li><a href="#Importing-Required-Libraries">Importing Required Libraries</a></li>
            <li><a href="#OpenRouter-LLM-Integration">OpenRouter LLM Integration</a></li>
            <li><a href="#Sample-Data-Setup">Sample Data Setup</a></li>
        </ol>
    </li>
    <li>
        <a href="#Background">Background</a>
        <ol>
            <li><a href="#What-are-Advanced-Retrievers?">What are Advanced Retrievers?</a></li>
            <li><a href="#Why-are-Advanced-Retrievers-Important?">Why are Advanced Retrievers Important?</a></li>
            <li><a href="#Index-Types-Overview">Index Types Overview</a></li>
        </ol>
    </li>
    <li>
        <a href="#Core-Retriever-Demonstrations">Core Retriever Demonstrations</a>
        <ol>
            <li><a href="#Vector-Index-Retriever">Vector Index Retriever</a></li>
            <li><a href="#BM25-Retriever">BM25 Retriever</a></li>
            <li><a href="#Document-Summary-Index-Retrievers">Document Summary Index Retrievers</a></li>
            <li><a href="#Auto-Merging-Retriever">Auto Merging Retriever</a></li>
            <li><a href="#Recursive-Retriever">Recursive Retriever</a></li>
            <li><a href="#Query-Fusion-Retriever">Query Fusion Retriever</a></li>
        </ol>
    </li>
    <li><a href="#Exercises">Exercises</a></li>
</ol>


## Objectives

After completing this lab you will be able to:

- Understand the different types of retrievers available in LlamaIndex and their use cases
- Implement Vector Index Retriever for semantic search
- Use BM25 Retriever for keyword-based search with advanced ranking
- Create Document Summary Index Retrievers for efficient large-scale retrieval
- Build Auto Merging Retriever for hierarchical context preservation
- Implement Recursive Retriever for multi-level reference following
- Apply Query Fusion Retriever with multiple fusion strategies (RRF, Relative Score, Distribution-Based)
- Integrate OpenRouter as the LLM backend for LlamaIndex


## Setup

For this lab, we will be using the following libraries:

*   [`llama-index`](https://docs.llamaindex.ai/) - The core LlamaIndex library for building RAG applications
*   [`llama-index-llms-openai`](https://docs.llamaindex.ai/en/stable/api_reference/llms/openai/) - OpenAI-compatible LLM integration (used with OpenRouter)
*   [`llama-index-embeddings-huggingface`](https://docs.llamaindex.ai/en/stable/api_reference/embeddings/huggingface/) - HuggingFace embeddings for local embedding generation
*   [`llama-index-retrievers-bm25`](https://docs.llamaindex.ai/en/stable/api_reference/retrievers/bm25/) - BM25 retriever implementation
*   [`python-dotenv`](https://pypi.org/project/python-dotenv/) - Load environment variables from `.env` file
*   [`sentence-transformers`](https://www.sbert.net/) - Sentence transformers for embeddings and reranking


### Installing Required Libraries

Run the following cell to install required libraries.

**NOTE**: The installation process takes about **5** minutes to complete.

```
  ( (
   ) )
........
|      |]
\      /   
 `----'
```


In [ ]:
# !pip install llama-index==0.12.49 \
#     llama-index-embeddings-huggingface==0.5.5 \
#     llama-index-llms-openai==0.4.0 \
#     llama-index-retrievers-bm25==0.5.2 \
#     sentence-transformers==5.0.0 \
#     rank-bm25==0.2.2 \
#     PyStemmer==2.2.0.3 \
#     python-dotenv==1.0.1 | tail -n 1

### Importing Required Libraries

We import all the necessary libraries for this lab, including core LlamaIndex components, retrievers, and OpenRouter integration via the OpenAI-compatible interface:


In [1]:
import os
import json
from typing import List, Optional
import asyncio
import warnings
import numpy as np
warnings.filterwarnings('ignore')

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Core LlamaIndex imports
from llama_index.core import (
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    Document,
    Settings,
    DocumentSummaryIndex,
    KeywordTableIndex
)
from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexRetriever,
    AutoMergingRetriever,
    RecursiveRetriever,
    QueryFusionRetriever
)
from llama_index.core.indices.document_summary import (
    DocumentSummaryIndexLLMRetriever,
    DocumentSummaryIndexEmbeddingRetriever,
)
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.embeddings import BaseEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import httpx

# OpenRouter / OpenAI-compatible LLM integration
from llama_index.llms.openrouter import OpenRouter

# Advanced retriever imports
from llama_index.retrievers.bm25 import BM25Retriever

# Sentence transformers
from sentence_transformers import SentenceTransformer

# Statistical libraries for fusion techniques
from scipy import stats

print("✅ All imports successful!")

I0000 00:00:1775570330.494439  124186 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775570331.713904  124186 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775570335.398249  124186 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ All imports successful!


## OpenRouter LLM Integration

We'll use [OpenRouter](https://openrouter.ai/) as the LLM provider. OpenRouter exposes an OpenAI-compatible API, so we can use the `llama-index-llms-openai` package with a custom `api_base` pointing to OpenRouter.

The credentials are loaded from the `.env` file:
- `OPENROUTER_API_KEY` — your OpenRouter API key
- `OPENROUTER_BASE_URL` — the OpenRouter base URL (or a custom proxy)


In [2]:
def create_openrouter_llm():
    """Create an OpenRouter LLM instance using llama-index-llms-openrouter."""
    api_key = os.getenv("OPENROUTER_API_KEY")
    base_url = os.getenv("OPENROUTER_BASE_URL")
    model_id = os.getenv("MODEL_ID")

    if not api_key or not base_url:
        raise ValueError("OPENROUTER_API_KEY and OPENROUTER_BASE_URL must be set in the .env file.")

    llm = OpenRouter(
        model=model_id,
        api_key=api_key,
        api_base=base_url,
        temperature=0.7,
        max_tokens=512,
        http_client=httpx.Client(verify=False)
    )
    print(f"✅ OpenRouter LLM initialized (model: {model_id})")
    return llm
   

In [3]:
import os

# Modèle local déjà disponible dans le projet (évite tout téléchargement)
LOCAL_EMBED_MODEL = os.path.join(
    os.path.dirname(os.getcwd()), "bge-small-en-v1.5"
)
# Fallback : utilise le chemin local si disponible, sinon tente HuggingFace
model_name = LOCAL_EMBED_MODEL if os.path.isdir(LOCAL_EMBED_MODEL) else "all-MiniLM-L6-v2"
print(f"🔧 Chargement du modèle d'embedding depuis : {model_name}")

embed_model = HuggingFaceEmbedding(model_name=model_name)
print("✅ HuggingFace embeddings initialisés !")

# Setup with OpenRouter
print("🔧 Initializing OpenRouter LLM...")
llm = create_openrouter_llm()

# Configure global settings
Settings.llm = llm
Settings.embed_model = embed_model
print("✅ OpenRouter LLM and embeddings configured!")


🔧 Chargement du modèle d'embedding depuis : /home/aelhidal/protos/Advanced-RAG-with-Vector-Databases-and-Retrievers/bge-small-en-v1.5
✅ HuggingFace embeddings initialisés !
🔧 Initializing OpenRouter LLM...
✅ OpenRouter LLM initialized (model: nvidia/nemotron-3-super-120b-a12b:free)
✅ OpenRouter LLM and embeddings configured!


---

## Background

Before diving into the advanced retrieval techniques, let's understand the foundational concepts that make these retrievers powerful.

### What are Advanced Retrievers?

Advanced retrievers in LlamaIndex are sophisticated components that go beyond simple vector similarity search to provide more accurate, context-aware, and efficient information retrieval.

### Why are Advanced Retrievers Important?

- **Better Precision**: Different retrieval methods excel in different scenarios
- **Context Preservation**: Hierarchical methods maintain document structure
- **Query Coverage**: Fusion methods handle diverse query formulations
- **Production Readiness**: Combining strategies improves robustness

### Index Types Overview

| Index Type | Best For | Retriever |
|---|---|---|
| VectorStoreIndex | Semantic search | VectorIndexRetriever |
| DocumentSummaryIndex | Large doc collections | DocumentSummaryIndexRetriever |
| KeywordTableIndex | Exact keyword matching | KeywordTableSimpleRetriever |
| Hierarchical Nodes | Long documents | AutoMergingRetriever |


## Initialize Lab Environment

Let's create our lab class and initialize all the indexes we'll need for different retrievers.


In [4]:
# Sample data for the lab - AI/ML focused documents
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

print(f"📄 Loaded {len(SAMPLE_DOCUMENTS)} sample documents")
print(f"🔍 Prepared {len(DEMO_QUERIES)} consistent demo queries")
for i, doc in enumerate(SAMPLE_DOCUMENTS[:3], 1):
    print(f"{i}. {doc}")
print("...")

📄 Loaded 10 sample documents
🔍 Prepared 7 consistent demo queries
1. Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
2. Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
3. Natural language processing enables computers to understand, interpret, and generate human language.
...


In [5]:
class AdvancedRetrieversLab:
    def __init__(self):
        print("🚀 Initializing Advanced Retrievers Lab...")
        self.documents = [Document(text=text) for text in SAMPLE_DOCUMENTS]
        self.nodes = SentenceSplitter().get_nodes_from_documents(self.documents)
        self._llm = create_openrouter_llm()
        
        print("📊 Creating indexes...")
        # Create various indexes
        self.vector_index = VectorStoreIndex.from_documents(self.documents)
        self.document_summary_index = DocumentSummaryIndex.from_documents(self.documents, llm=self._llm)
        self.keyword_index = KeywordTableIndex.from_documents(self.documents)
        
        print("✅ Advanced Retrievers Lab Initialized!")
        print(f"📄 Loaded {len(self.documents)} documents")
        print(f"🔢 Created {len(self.nodes)} nodes")

# Initialize the lab
lab = AdvancedRetrieversLab()

🚀 Initializing Advanced Retrievers Lab...
✅ OpenRouter LLM initialized (model: nvidia/nemotron-3-super-120b-a12b:free)
📊 Creating indexes...
current doc id: 4ebc7160-605d-4612-85f0-33327b53c6d3
current doc id: 4f2f10a5-d4df-4784-99e7-daae23084396
current doc id: 5d36947f-546d-42cb-855c-0721b71e1f0c
current doc id: b1af2363-3b5b-4d92-bcd0-fe3ee84f4b7a
current doc id: 561563e9-e511-4b91-bb05-baca13ad22d5
current doc id: 06740f9d-b7fa-44b8-b690-803858992963
current doc id: 57147e5e-b144-4bed-89d3-d20e80f7774e
current doc id: 435c9e24-d1d9-461a-86fe-cd4708a79285
current doc id: 0b7ec86e-df28-4b3e-abcb-1086f9750192
current doc id: d390cf65-1ead-4cfb-bdb2-315e91d88b3f
✅ Advanced Retrievers Lab Initialized!
📄 Loaded 10 documents
🔢 Created 10 nodes


In [8]:
lab.nodes[0].text

'Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.'

---

# Core Retriever Demonstrations

This lab focuses on the essential retrievers in LlamaIndex, covering core retrieval methods, advanced retrievers, and fusion techniques. Each section provides practical examples and detailed explanations based on official LlamaIndex documentation.


## 1. Vector Index Retriever - The Foundation

The Vector Index Retriever uses vector embeddings to find semantically related content, making it ideal for general-purpose search and widely used in retrieval-augmented generation (RAG) pipelines.

**How it works**: 
- Documents are split into nodes and encoded as dense vectors using an embedding model
- At query time, the query is embedded and compared to all node vectors via cosine similarity
- The top-k most similar nodes are returned


In [9]:
print("=" * 60)
print("1. VECTOR INDEX RETRIEVER")
print("=" * 60)

# Basic vector retriever
vector_retriever = VectorIndexRetriever(
    index=lab.vector_index,
    similarity_top_k=3
)

# Alternative creation method
alt_retriever = lab.vector_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES["basic"]  # "What is machine learning?"
nodes = vector_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Retrieved {len(nodes)} nodes:")
for i, node in enumerate(nodes, 1):
    print(f"{i}. Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

1. VECTOR INDEX RETRIEVER
Query: What is machine learning?
Retrieved 3 nodes:
1. Score: 0.8792
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.7651
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Score: 0.7084
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...



## 2. BM25 Retriever - Advanced Keyword-Based Search

BM25 is a keyword-based retrieval method that improves on TF-IDF by addressing some of its key limitations. It's widely used in production search systems including Elasticsearch and Apache Lucene.

**Key improvements over TF-IDF:**
- **Term frequency saturation**: Prevents over-scoring repeated terms
- **Document length normalization**: Prevents long document bias
- **Tunable parameters**: k1 (saturation ≈ 1.2) and b (length normalization ≈ 0.75)


In [10]:
print("=" * 60)
print("2. BM25 RETRIEVER")
print("=" * 60)

import Stemmer

# Create BM25 retriever with default parameters
bm25_retriever = BM25Retriever.from_defaults(
    nodes=lab.nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)

query = DEMO_QUERIES["technical"]  # "neural networks deep learning"
nodes = bm25_retriever.retrieve(query)

print(f"Query: {query}")
print("BM25 analyzes exact keyword matches with sophisticated scoring")
print(f"Retrieved {len(nodes)} nodes:")

for i, node in enumerate(nodes, 1):
    score = node.score if hasattr(node, 'score') and node.score else 0
    print(f"{i}. BM25 Score: {score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    
    # Highlight which query terms appear in the text
    text_lower = node.text.lower()
    query_terms = query.lower().split()
    found_terms = [term for term in query_terms if term in text_lower]
    if found_terms:
        print(f"   → Found terms: {found_terms}")
    print()

print("BM25 vs TF-IDF Comparison:")
print("TF-IDF Problem: Linear term frequency scaling")
print("  Example: 10 occurrences → score of 10, 100 occurrences → score of 100")
print("BM25 Solution: Saturation function")
print("  Example: 10 occurrences → high score, 100 occurrences → slightly higher score")
print()
print("Key BM25 Parameters:")
print("- k1 ≈ 1.2: Term frequency saturation (how quickly scores plateau)")
print("- b ≈ 0.75: Document length normalization (0=none, 1=full)")
print("- IDF weighting: Rare terms get higher scores")

2. BM25 RETRIEVER
Query: neural networks deep learning
BM25 analyzes exact keyword matches with sophisticated scoring
Retrieved 3 nodes:
1. BM25 Score: 2.5203
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...
   → Found terms: ['neural', 'networks', 'deep', 'learning']

2. BM25 Score: 0.3372
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...
   → Found terms: ['learning']

3. BM25 Score: 0.3024
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...
   → Found terms: ['learning']

BM25 vs TF-IDF Comparison:
TF-IDF Problem: Linear term frequency scaling
  Example: 10 occurrences → score of 10, 100 occurrences → score of 100
BM25 Solution: Saturation function
  Example: 10 occurrences → high score, 100 occurrences → slightly higher score

Key BM25 Parameters:
- k1 ≈ 1.2: Term frequency saturation (how quickly

## 3. Document Summary Index Retrievers

Document Summary Index Retrievers use document summaries instead of the actual documents to find relevant content, making them efficient for large collections. **They return the original documents, not their summaries.**

**How it works**: The LLM generates a summary for each document during indexing. At retrieval time, the query is matched against summaries (via LLM or embeddings) to select relevant documents.


In [11]:
print("=" * 60)
print("3. DOCUMENT SUMMARY INDEX RETRIEVERS")
print("=" * 60)

# LLM-based document summary retriever
doc_summary_retriever_llm = DocumentSummaryIndexLLMRetriever(
    lab.document_summary_index,
    choice_top_k=3  # Number of documents to select
)

# Embedding-based document summary retriever  
doc_summary_retriever_embedding = DocumentSummaryIndexEmbeddingRetriever(
    lab.document_summary_index,
    similarity_top_k=3  # Number of documents to select
)

query = DEMO_QUERIES["learning_types"]  # "different types of learning"

print(f"Query: {query}")

print("\nA) LLM-based Document Summary Retriever:")
print("Uses LLM to select relevant documents based on summaries")
try:
    nodes_llm = doc_summary_retriever_llm.retrieve(query)
    print(f"Retrieved {len(nodes_llm)} nodes")
    for i, node in enumerate(nodes_llm[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"LLM-based retrieval demo: {str(e)[:100]}...")

print("B) Embedding-based Document Summary Retriever:")
print("Uses vector similarity between query and document summaries")
try:
    nodes_emb = doc_summary_retriever_embedding.retrieve(query)
    print(f"Retrieved {len(nodes_emb)} nodes")
    for i, node in enumerate(nodes_emb[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"Embedding-based retrieval demo: {str(e)[:100]}...")

print("Document Summary Index workflow:")
print("1. Generates summaries for each document using LLM")
print("2. Uses summaries to select relevant documents")
print("3. Returns full content from selected documents")

3. DOCUMENT SUMMARY INDEX RETRIEVERS
Query: different types of learning

A) LLM-based Document Summary Retriever:
Uses LLM to select relevant documents based on summaries
Retrieved 3 nodes
1. Score: 10.0000
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to...

2. Score: 9.0000
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....

B) Embedding-based Document Summary Retriever:
Uses vector similarity between query and document summaries
Retrieved 3 nodes
1. (Document summary)
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....

2. (Document summary)
   Text: Transfer learning leverages knowledge from pre-trained models to improve perform...

Document Summary Index workflow:
1. Generates summaries for each document using LLM
2. Uses summaries to select relevant documents
3. Returns full content from selected documents


## 4. Auto Merging Retriever - Hierarchical Context Preservation

Auto Merging Retriever is designed to preserve context in long documents using a hierarchical structure. **It uses hierarchical chunking to break documents into parent and child nodes, and if enough child nodes from the same parent are retrieved, it merges them back into the parent node.**

**How it works**: Documents are chunked at multiple granularities (e.g., 512, 256, 128 tokens). Child nodes are retrieved first; if a sufficient fraction of a parent's children are retrieved, the parent replaces them, providing richer context.


In [12]:
print("=" * 60)
print("4. AUTO MERGING RETRIEVER")
print("=" * 60)

# Create hierarchical nodes
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512, 256, 128]
)

hier_nodes = node_parser.get_nodes_from_documents(lab.documents)

# Create storage context with all nodes
from llama_index.core import StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.vector_stores import SimpleVectorStore

docstore = SimpleDocumentStore()
docstore.add_documents(hier_nodes)

storage_context = StorageContext.from_defaults(docstore=docstore)

# Create base index
base_index = VectorStoreIndex(hier_nodes, storage_context=storage_context)
base_retriever = base_index.as_retriever(similarity_top_k=6)

# Create auto-merging retriever
auto_merging_retriever = AutoMergingRetriever(
    base_retriever, 
    storage_context,
    verbose=True
)

query = DEMO_QUERIES["advanced"]  # "How do neural networks work in deep learning?"
nodes = auto_merging_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Auto-merged to {len(nodes)} nodes")
for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Auto-merged)")
    print(f"   Text: {node.text[:120]}...")
    print()

4. AUTO MERGING RETRIEVER
> Merging 1 nodes into parent node.
> Parent node id: be429aa0-2855-4ce4-a4df-b486c26671e3.
> Parent node text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs.

> Merging 1 nodes into parent node.
> Parent node id: 1d5d3a44-0d43-49f0-9272-68e36fa223d2.
> Parent node text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs.

Query: How do neural networks work in deep learning?
Auto-merged to 2 nodes
1. Score: 0.8569
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data....

2. Score: 0.7128
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....



## 5. Recursive Retriever - Multi-Level Reference Following

The Recursive Retriever is **designed to follow relationships between nodes using references**. It can follow references from one node to another, such as citations in academic papers or other metadata links, allowing it to **retrieve related content across multiple hops**.


In [13]:
print("=" * 60)
print("5. RECURSIVE RETRIEVER")
print("=" * 60)

# Create documents with references
docs_with_refs = []
for i, doc in enumerate(lab.documents):
    ref_doc = Document(
        text=doc.text,
        metadata={
            "doc_id": f"doc_{i}",
            "references": [f"doc_{j}" for j in range(len(lab.documents)) if j != i][:2]
        }
    )
    docs_with_refs.append(ref_doc)

# Create index with referenced documents
ref_index = VectorStoreIndex.from_documents(docs_with_refs)

# Create retriever mapping
retriever_dict = {
    f"doc_{i}": ref_index.as_retriever(similarity_top_k=1)
    for i in range(len(docs_with_refs))
}

# Base retriever
base_retriever = ref_index.as_retriever(similarity_top_k=2)

# Add the root retriever to the dictionary
retriever_dict["vector"] = base_retriever

# Recursive retriever
recursive_retriever = RecursiveRetriever(
    "vector",
    retriever_dict=retriever_dict,
    query_engine_dict={},
    verbose=True
)

query = DEMO_QUERIES["applications"]  # "What are the applications of AI?"
try:
    nodes = recursive_retriever.retrieve(query)
    print(f"Query: {query}")
    print(f"Recursively retrieved {len(nodes)} nodes")
    for i, node in enumerate(nodes[:3], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Recursive)")
        print(f"   Text: {node.text[:100]}...")
        print()
except Exception as e:
    print(f"Query: {query}")
    print(f"Recursive retriever demo: {str(e)}")
    print("Note: Recursive retriever requires specific node reference setup")
    
    # Fallback to basic retrieval for demonstration
    print("\nFalling back to basic retrieval demonstration...")
    base_nodes = base_retriever.retrieve(query)
    for i, node in enumerate(base_nodes[:2], 1):
        print(f"{i}. Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()

5. RECURSIVE RETRIEVER
Retrieving with query id None: What are the applications of AI?
Retrieving text node: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
Retrieving text node: Generative AI can create new content including text, images, code, and more.
Query: What are the applications of AI?
Recursively retrieved 2 nodes
1. Score: 0.6656
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.6292
   Text: Generative AI can create new content including text, images, code, and more....



## 6. Query Fusion Retriever - Multi-Query Enhancement with Advanced Fusion

The Query Fusion Retriever **combines results from different retrievers** and **optionally generates multiple variations of a query using an LLM to improve coverage**. It supports three fusion modes:

1. **`reciprocal_rerank`** (RRF) — Most robust, rank-based, scale-invariant
2. **`relative_score`** — Preserves confidence, normalizes by max score
3. **`dist_based_score`** — Most sophisticated, statistical normalization


In [14]:
print("=" * 60)
print("6. QUERY FUSION RETRIEVER - OVERVIEW")
print("=" * 60)

# Create base retriever
base_retriever = lab.vector_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"
print(f"Query: {query}")
print("QueryFusionRetriever generates multiple query variations and fuses results")
print("using one of three sophisticated fusion modes.")

print("\nOverview of Fusion Modes:")
print("1. RECIPROCAL_RERANK: Uses reciprocal rank fusion (most robust)")
print("2. RELATIVE_SCORE: Preserves score magnitudes (most interpretable)")  
print("3. DIST_BASED_SCORE: Statistical normalization (most sophisticated)")

print("\nProceed to subsections 6.1, 6.2, and 6.3 for detailed demonstrations...")

6. QUERY FUSION RETRIEVER - OVERVIEW
Query: What are the main approaches to machine learning?
QueryFusionRetriever generates multiple query variations and fuses results
using one of three sophisticated fusion modes.

Overview of Fusion Modes:
1. RECIPROCAL_RERANK: Uses reciprocal rank fusion (most robust)
2. RELATIVE_SCORE: Preserves score magnitudes (most interpretable)
3. DIST_BASED_SCORE: Statistical normalization (most sophisticated)

Proceed to subsections 6.1, 6.2, and 6.3 for detailed demonstrations...


### 6.1 Reciprocal Rank Fusion (RRF) Mode

Reciprocal Rank Fusion is the most robust fusion method in QueryFusionRetriever, designed to combine ranked lists from multiple query variations by using the reciprocal of ranks, which reduces the impact of outliers and provides stable fusion results.

**Formula**: `RRF_score = Σ 1 / (rank + k)` where k=60 is a smoothing constant.


In [15]:
print("=" * 60)
print("6.1 RECIPROCAL RANK FUSION MODE DEMONSTRATION")
print("=" * 60)

base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    rrf_query_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="reciprocal_rerank",
        use_async=False,
        verbose=True
    )
    
    print(f"\nQuery: {query}")
    print("QueryFusionRetriever will:")
    print("1. Generate query variations using OpenRouter LLM")
    print("2. Retrieve results for each variation")
    print("3. Apply Reciprocal Rank Fusion")
    
    nodes = rrf_query_fusion.retrieve(query)
    
    print(f"\nRRF Query Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Final RRF Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("RRF Benefits in Query Fusion Context:")
    print("- Automatically handles query variations of different quality")
    print("- No bias toward queries that return higher raw scores")
    print("- Stable performance across diverse query formulations")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating RRF concept manually with query variations...")
    
    query_variations = [
        DEMO_QUERIES["comprehensive"],
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    all_results = {}
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        for rank, node in enumerate(nodes):
            node_id = node.node.node_id
            if node_id not in all_results:
                all_results[node_id] = {'node': node, 'rrf_score': 0, 'query_ranks': []}
            k = 60
            rrf_contribution = 1.0 / (rank + 1 + k)
            all_results[node_id]['rrf_score'] += rrf_contribution
            all_results[node_id]['query_ranks'].append((i, rank + 1))
    
    sorted_results = sorted(all_results.values(), key=lambda x: x['rrf_score'], reverse=True)
    
    print(f"\nCombined RRF Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Final RRF Score: {result['rrf_score']:.4f}")
        print(f"   Query ranks: {result['query_ranks']}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()
    
    print("RRF Formula: score = Σ(1 / (rank + 60))")
    print("- Rank 1: 1/(1+60) = 0.0164")
    print("- Rank 2: 1/(2+60) = 0.0161")
    print("Documents appearing in multiple queries get higher combined scores")

6.1 RECIPROCAL RANK FUSION MODE DEMONSTRATION

Query: What are the main approaches to machine learning?
QueryFusionRetriever will:
1. Generate query variations using OpenRouter LLM
2. Retrieve results for each variation
3. Apply Reciprocal Rank Fusion
Generated queries:
What are the main approaches to machine learning: supervised, unsupervised, and reinforcement learning?
Overview of machine learning paradigms and their typical applications.

RRF Query Fusion Results:
1. Final RRF Score: 0.0497
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Final RRF Score: 0.0484
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

3. Final RRF Score: 0.0481
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

RRF Benefits in Query Fusion Context:
- Automatically handles query variations of different quality
- No bias towar

### 6.2 Relative Score Fusion Mode

Relative Score Fusion normalizes retrieval scores relative to the maximum score within each query variation's results, enabling effective combination when you want to preserve score magnitude information across different query formulations.

**Formula**: `normalized_score = original_score / max_score` per query variation.


In [43]:
print("=" * 60)
print("6.2 RELATIVE SCORE FUSION MODE DEMONSTRATION")
print("=" * 60)

base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)
query = DEMO_QUERIES["comprehensive"]

try:
    rel_score_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="relative_score",
        use_async=False,
        verbose=False
    )
    
    print(f"\nQuery: {query}")
    nodes = rel_score_fusion.retrieve(query)
    
    print(f"\nRelative Score Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Combined Relative Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("Relative Score Benefits:")
    print("- Preserves confidence information from the embedding model")
    print("- Ensures fair contribution from each query variation")
    print("- More interpretable than rank-only methods")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating Relative Score concept manually...")
    
    query_variations = [
        DEMO_QUERIES["comprehensive"],
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    all_results = {}
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        scores = [node.score or 0 for node in nodes]
        max_score = max(scores) if scores else 1.0
        print(f"Max score: {max_score:.4f}")
        
        for node in nodes:
            node_id = node.node.node_id
            original_score = node.score or 0
            normalized_score = original_score / max_score if max_score > 0 else 0
            
            if node_id not in all_results:
                all_results[node_id] = {'node': node, 'combined_score': 0, 'contributions': []}
            
            all_results[node_id]['combined_score'] += normalized_score
            all_results[node_id]['contributions'].append({'query': i, 'original': original_score, 'normalized': normalized_score})
    
    sorted_results = sorted(all_results.values(), key=lambda x: x['combined_score'], reverse=True)
    
    print(f"\nCombined Relative Score Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Combined Score: {result['combined_score']:.4f}")
        for contrib in result['contributions']:
            print(f"     Query {contrib['query']}: {contrib['original']:.3f} → {contrib['normalized']:.3f}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()

6.2 RELATIVE SCORE FUSION MODE DEMONSTRATION

Query: What are the main approaches to machine learning?

Relative Score Fusion Results:
1. Combined Relative Score: 0.6667
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Combined Relative Score: 0.4187
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

3. Combined Relative Score: 0.4043
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

Relative Score Benefits:
- Preserves confidence information from the embedding model
- Ensures fair contribution from each query variation
- More interpretable than rank-only methods


### 6.3 Distribution-Based Score Fusion Mode

Distribution-Based Score Fusion uses statistical properties of score distributions from each query variation to normalize and combine retrieval results, providing the most sophisticated handling of score variability.

**Formula**: Z-score normalization → sigmoid transform → summed across variations.


In [16]:
print("=" * 60)
print("6.3 DISTRIBUTION-BASED SCORE FUSION MODE DEMONSTRATION")
print("=" * 60)

base_retriever = lab.vector_index.as_retriever(similarity_top_k=8)
query = DEMO_QUERIES["comprehensive"]

try:
    dist_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="dist_based_score",
        use_async=False,
        verbose=False
    )
    
    print(f"\nQuery: {query}")
    nodes = dist_fusion.retrieve(query)
    
    print(f"\nDistribution-Based Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Statistically Normalized Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("Distribution-Based Benefits:")
    print("- Accounts for score distribution differences between query variations")
    print("- Statistically robust against outliers and noise")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating Distribution-Based concept manually...")
    
    query_variations = [
        DEMO_QUERIES["comprehensive"],
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    all_results = {}
    
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        scores = [node.score or 0 for node in nodes]
        
        mean_score = np.mean(scores) if scores else 0
        std_score = np.std(scores) if len(scores) > 1 else 1
        
        print(f"Distribution: mean={mean_score:.3f}, std={std_score:.3f}")
        
        for node, score in zip(nodes, scores):
            node_id = node.node.node_id
            z_score = (score - mean_score) / std_score if std_score > 0 else 0
            normalized_score = 1 / (1 + np.exp(-z_score))  # sigmoid
            
            if node_id not in all_results:
                all_results[node_id] = {'node': node, 'combined_score': 0, 'contributions': []}
            
            all_results[node_id]['combined_score'] += normalized_score
            all_results[node_id]['contributions'].append({'query': i, 'original': score, 'z_score': z_score, 'normalized': normalized_score})
    
    sorted_results = sorted(all_results.values(), key=lambda x: x['combined_score'], reverse=True)
    
    print(f"\nCombined Distribution-Based Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Combined Score: {result['combined_score']:.4f}")
        for contrib in result['contributions']:
            print(f"     Q{contrib['query']}: {contrib['original']:.3f} → z={contrib['z_score']:.2f} → {contrib['normalized']:.3f}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()

# Summary comparison
print("\n" + "=" * 60)
print("FUSION MODES COMPARISON SUMMARY")
print("=" * 60)
print(f"Query used across all modes: '{DEMO_QUERIES['comprehensive']}'")
print()
print("• RRF (reciprocal_rerank): Most robust, rank-based, scale-invariant")
print("• Relative Score: Preserves confidence, normalizes by max score")  
print("• Distribution-Based: Most sophisticated, statistical normalization")
print()
print("Choose based on your use case:")
print("- Production stability → RRF")
print("- Score interpretability → Relative Score")
print("- Statistical robustness → Distribution-Based")

6.3 DISTRIBUTION-BASED SCORE FUSION MODE DEMONSTRATION

Query: What are the main approaches to machine learning?

Distribution-Based Fusion Results:
1. Statistically Normalized Score: 0.8221
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Statistically Normalized Score: 0.5773
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

3. Statistically Normalized Score: 0.5511
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

Distribution-Based Benefits:
- Accounts for score distribution differences between query variations
- Statistically robust against outliers and noise

FUSION MODES COMPARISON SUMMARY
Query used across all modes: 'What are the main approaches to machine learning?'

• RRF (reciprocal_rerank): Most robust, rank-based, scale-invariant
• Relative Score: Preserves confidence, normalizes by max sco

## Recommended Retrievers by Use Case

**General Q&A Applications:**
- **Primary**: Vector Index Retriever for semantic understanding
- **Enhancement**: Combine with BM25 for hybrid search

**Large Document Collections:**
- **Primary**: Document Summary Index for efficient selection
- **Context**: Auto Merging for preserving document structure

**Multi-Query Coverage:**
- **Fusion**: Query Fusion Retriever with RRF mode
- **Stability**: RRF is the safest default for production

**Reference-Heavy Content:**
- **Primary**: Recursive Retriever for following citations


---

# Exercises

Now that you've learned about advanced retrievers, let's practice implementing them in different scenarios.


## Exercise 1 - Build a Custom Hybrid Retriever

Your task is to create a hybrid retriever that combines both vector similarity and BM25 keyword search for improved results.

**Requirements:**
- Use both Vector Index Retriever and BM25 Retriever
- Implement a simple score fusion mechanism which takes the average of both scores
- Return the top 5 combined results

```python
# TODO: Implement hybrid retriever
def hybrid_retrieve(query, top_k=5):
    # Get results from both retrievers
    # Combine and re-rank
    pass
```


In [18]:
### Put your solution here ###
def hybrid_retrieve(query: list, top_k: int=5):
    vector_index_retriever = VectorIndexRetriever(
        index=lab.vector_index,
        similarity_top_k=10
        )

    bm25_retriever = BM25Retriever.from_defaults(
        nodes=lab.nodes,
        similarity_top_k=10,
        stemmer=Stemmer.Stemmer("english"),
        language="english"
    )

    vector_index_retriever_nodes = vector_index_retriever.retrieve(query)
    bm25_retriever_nodes = bm25_retriever.retrieve(query)

    
    combined = {}
    for node in vector_index_retriever_nodes:
        nid = node.node.node_id
        combined[nid] = {'node': node, 'vector_score': node.score or 0, 'bm25_score': 0}
    for node in bm25_retriever_nodes:
        nid = node.node.node_id
        if nid in combined:
            combined[nid]['bm25_score'] = node.score or 0
        else:
            combined[nid] = {'node': node, 'vector_score': 0, 'bm25_score': node.score or 0}
    
    for v in combined.values():
        v['hybrid_score'] = (v['vector_score'] + v['bm25_score']) / 2
    
    sorted_results = sorted(combined.values(), key=lambda x: x['hybrid_score'], reverse=True)
    return sorted_results[:top_k]

results = hybrid_retrieve("What is machine learning?")
for i, r in enumerate(results, 1):
    print(f"{i}. Hybrid: {r['hybrid_score']:.4f} (vec={r['vector_score']:.3f}, bm25={r['bm25_score']:.3f})")
    print(f"   {r['node'].text[:80]}...")



1. Hybrid: 0.4396 (vec=0.879, bm25=0.000)
   Machine learning is a subset of artificial intelligence that focuses on algorith...
2. Hybrid: 0.3825 (vec=0.765, bm25=0.000)
   Reinforcement learning is a type of machine learning where agents learn to make ...
3. Hybrid: 0.3822 (vec=0.000, bm25=0.764)
   Reinforcement learning is a type of machine learning where agents learn to make ...
4. Hybrid: 0.3821 (vec=0.000, bm25=0.764)
   Machine learning is a subset of artificial intelligence that focuses on algorith...
5. Hybrid: 0.3542 (vec=0.708, bm25=0.000)
   Deep learning uses neural networks with multiple layers to model and understand ...


<details>
    <summary>Click here for Solution</summary>

```python
# Create both retrievers
vector_retriever = lab.vector_index.as_retriever(similarity_top_k=10)
try:
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=lab.nodes, similarity_top_k=10
    )
except:
    bm25_retriever = vector_retriever

def hybrid_retrieve(query, top_k=5):
    vector_results = vector_retriever.retrieve(query)
    bm25_results = bm25_retriever.retrieve(query)
    
    combined = {}
    for node in vector_results:
        nid = node.node.node_id
        combined[nid] = {'node': node, 'vector_score': node.score or 0, 'bm25_score': 0}
    for node in bm25_results:
        nid = node.node.node_id
        if nid in combined:
            combined[nid]['bm25_score'] = node.score or 0
        else:
            combined[nid] = {'node': node, 'vector_score': 0, 'bm25_score': node.score or 0}
    
    for v in combined.values():
        v['hybrid_score'] = (v['vector_score'] + v['bm25_score']) / 2
    
    sorted_results = sorted(combined.values(), key=lambda x: x['hybrid_score'], reverse=True)
    return sorted_results[:top_k]

results = hybrid_retrieve("What is machine learning?")
for i, r in enumerate(results, 1):
    print(f"{i}. Hybrid: {r['hybrid_score']:.4f} (vec={r['vector_score']:.3f}, bm25={r['bm25_score']:.3f})")
    print(f"   {r['node'].text[:80]}...")
```
</details>


## Exercise 2 - Create a Production RAG Pipeline

Build a complete RAG pipeline that uses multiple retrieval strategies and includes evaluation metrics.

**Requirements:**
- Implement retrieval with multiple strategies
- Add query routing logic
- Include basic evaluation metrics that evaluate whether the pipeline succeeded or failed
- Handle edge cases and errors

```python
# TODO: Implement production RAG pipeline
class ProductionRAGPipeline:
    def __init__(self, index, llm):
        self.index = index
        self.llm = llm
        # Add your implementation
```


In [ ]:
### Put your solution here ###

<details>
    <summary>Click here for Solution</summary>

```python
class ProductionRAGPipeline:
    def __init__(self, index, llm):
        self.index = index
        self.llm = llm
        self.vector_retriever = index.as_retriever(similarity_top_k=5)
        
    def _route_query(self, question):
        """Simple query routing based on question characteristics"""
        if any(word in question.lower() for word in ["what", "explain", "describe"]):
            return "semantic"
        elif any(word in question.lower() for word in ["list", "types", "examples"]):
            return "keyword"
        return "hybrid"
    
    def retrieve_and_answer(self, question):
        strategy = self._route_query(question)
        print(f"Strategy: {strategy}")
        
        try:
            nodes = self.vector_retriever.retrieve(question)
            context = "\n".join([n.text for n in nodes[:3]])
            
            query_engine = self.index.as_query_engine()
            response = query_engine.query(question)
            
            return {
                "answer": str(response),
                "nodes_retrieved": len(nodes),
                "strategy": strategy,
                "success": True
            }
        except Exception as e:
            return {"answer": None, "error": str(e), "success": False}

pipeline = ProductionRAGPipeline(lab.vector_index, llm)
result = pipeline.retrieve_and_answer("What is deep learning?")
print(f"Success: {result['success']}")
print(f"Answer: {result.get('answer', 'N/A')[:200]}")
```
</details>


## Summary

Congratulations! You've successfully learned about advanced retrievers in LlamaIndex using OpenRouter as the LLM backend. Here's what you've accomplished:

**Key Concepts Mastered:**
- **Vector Index Retriever**: Semantic search using embeddings
- **BM25 Retriever**: Advanced keyword-based search with TF-IDF improvements
- **Document Summary Index**: Intelligent document selection using summaries
- **Auto Merging Retriever**: Hierarchical context preservation
- **Recursive Retriever**: Multi-level reference following
- **Query Fusion Retriever**: Multi-query enhancement with RRF, Relative Score, and Distribution-Based fusion
- **OpenRouter Integration**: Using OpenAI-compatible API with custom base URL and API key from `.env`

**Next Steps:**
- Experiment with different OpenRouter models (e.g., `anthropic/claude-3-5-haiku`, `meta-llama/llama-3.1-8b-instruct`)
- Try combining retrievers with reranking using `SentenceTransformerRerank`
- Build a full RAG pipeline with evaluation metrics
